# Olist delivery intelligence

Explore the same original-data results used by the dashboard. Build the project with `python run_project.py` before running cells. The included outputs were generated from the verified local build.


In [1]:
from pathlib import Path
import sys
import duckdb
import pandas as pd

PROJECT = Path.cwd()
if not (PROJECT / "src").exists():
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))
from src import analysis as a
connection = duckdb.connect(str(PROJECT / "data/processed/olist.duckdb"), read_only=True)
orders = a.load_orders()


## 1. Establish the population before calculating a rate

Delivered, canceled and in-progress orders remain distinguishable. Only eligible completed deliveries enter the SLA denominator.


In [2]:
connection.sql("SELECT order_status, count(*) AS orders FROM analytics.fact_orders GROUP BY order_status ORDER BY orders DESC").df()


order_status,orders
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


## 2. Show why table grain matters

The naive items × payments join changes the item total. The production model separately aggregates each child table before joining.


In [3]:
query = (PROJECT / "sql/03_investigation.sql").read_text(encoding="utf-8").split(";")[0]
connection.execute(query).fetchdf()


method,item_value_brl
correct,13591643.70
naive_items_x_payments,14209115.34


## 3. Inspect the headline metrics

The promise is a calendar date; a delivery later on the same date counts as on time.


In [4]:
pd.DataFrame([a.summary(orders)]).T.reset_index().rename(columns={"index":"metric", 0:"value"})


metric,value
orders,9.944100e+04
delivered_orders,9.647800e+04
eligible_orders,9.647000e+04
late_orders,6.534000e+03
late_rate_pct,6.773090e+00
median_delivery_days,1.021748e+01
delivered_item_value_brl,1.322150e+07
multi_seller_eligible,1.275000e+03
reviewed_eligible,9.582400e+04
on_time_reviewed,8.944300e+04


## 4. Check the interpretation against review timing

Compare the full reviewed cohort with answers submitted on/after delivery. This is a sensitivity check, not a causal estimate.


In [5]:
pd.concat([
    a.review_comparison(orders).assign(scope="All reviewed"),
    a.review_comparison(orders, post_delivery_only=True).assign(scope="Answered on/after delivery"),
], ignore_index=True)


delivery,eligible_orders,reviewed_orders,poor_reviews,poor_review_pct,scope
On time,89936,89443,8289,9.267355,All reviewed
Late,6534,6381,3983,62.419683,All reviewed
On time,89263,89263,8258,9.251314,Answered on/after delivery
Late,1908,1908,371,19.444444,Answered on/after delivery


## 5. Prioritize an investigation

Rank meaningful late-order volume alongside minimum sample size and pointwise Wilson intervals. Check route and product mix before attributing responsibility.


In [6]:
a.sellers(orders, minimum=30)[["seller_id", "seller_state", "eligible_orders", "late_orders", "late_rate_pct", "wilson_lower_pct", "excess_late_orders", "investigate"]].head(10)


seller_id,seller_state,eligible_orders,late_orders,late_rate_pct,wilson_lower_pct,excess_late_orders,investigate
4a3ca9315b744ce9f8e9374361493884,SP,1673,168,10.041841,8.691885,53.396996,True
06a2c3af7b3aee5d69171b0e14f0ee87,MA,388,74,19.072165,15.473579,47.421419,True
4869f7a5dfa277a7dca6462dcf3b52b2,SP,1111,118,10.621062,8.943038,41.894837,True
1f50f920176fa81dab994f9023523100,SP,1364,124,9.090909,7.677966,30.563958,True
8160255418d5aaa7dbdc9f4c64ebda44,SP,355,51,14.366197,11.098588,26.681969,True
7d13fca15225358621be4086e1eb0964,SP,548,64,11.678832,9.252688,26.461180,True
88460e8ebdecbfecb5f9601833981930,PR,227,42,18.502203,13.989653,26.450160,True
7c67e1448b00f6e969d365cea6b010ab,SP,963,89,9.241952,7.571141,23.033058,True
ea8482cd71df3c1969d7b9473ff13abc,SP,1116,96,8.602151,7.095749,19.552329,True
e5a3438891c0bfdb9394643f95273d8e,SP,213,34,15.962441,11.653341,19.409181,True


In [7]:
a.routes(orders, minimum=100).head(10)


seller_state,customer_state,route,eligible_orders,late_orders,late_rate_pct,median_handling_days,median_transit_days
SP,SP,SP -> SP,30313,1425,4.700953,2.166024,3.556296
SP,RJ,SP -> RJ,8048,1147,14.251988,2.326800,9.120243
SP,MG,SP -> MG,7326,391,5.337155,2.259873,7.336042
SP,BA,SP -> BA,2278,297,13.037752,2.349120,14.117911
SP,RS,SP -> RS,3540,238,6.723164,2.318704,10.791962
SP,SC,SP -> SC,2306,222,9.627060,2.270382,10.204468
SP,ES,SP -> ES,1444,180,12.465374,2.427894,9.946042
SP,PR,SP -> PR,3077,138,4.484888,2.310498,7.765532
SP,CE,SP -> CE,961,133,13.839750,2.224861,14.962778
PR,RJ,PR -> RJ,951,117,12.302839,2.271227,10.118819


In [8]:
connection.close()
